In [73]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display
import warnings

import ast

warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

# 전역 시드 설정 (재현성을 위해)
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [74]:
# portfolio = pd.read_csv('../1data/portfolio.csv')
# profile = pd.read_csv('../1data/profile.csv')
# transcript = pd.read_csv('../1data/transcript.csv')

# merged = pd.read_csv('../1data/all_merged.csv')
# promotion = pd.read_csv('../1data/promotion_df.csv')
# normal = pd.read_csv('../1data/normal_flow_df.csv')

portfolio = pd.read_csv('../data/portfolio.csv')
profile = pd.read_csv('../data/profile.csv')
transcript = pd.read_csv('../data/transcript.csv')

# merged = pd.read_csv('../data/all_merged.csv')
# promotion = pd.read_csv('../data/promotion_df.csv')
# normal = pd.read_csv('../data/normal_flow_df.csv')

In [75]:
def check(df, name="Data"):
    print(f"{name} 기본 정보")
    
    print("\n[1] 데이터 크기")
    display(df.shape)
    
    print("\n[2] 컬럼 정보")
    df.info()
    
    print("\n[3] 결측치 개수")
    display(
    df.isnull().sum()
    .sort_values(ascending=False)
    .to_frame("결측치 개수")
    )
    
    print("\n[4] 중복 데이터 개수")
    display(df.duplicated().sum())

---

# profile

In [76]:
# 데이터 타입 date형식으로 변환
profile["became_member_on"] = pd.to_datetime(profile["became_member_on"], format="%Y%m%d")

In [77]:
# 필요없는 Unnamed:0 컬럼 제거
profile = profile.drop('Unnamed: 0', axis=1)

---

# portfolio

In [78]:
# channels마다 파생변수 생성
portfolio['web'] = portfolio['channels'].astype(str).str.contains('web').astype(int)
portfolio['email'] = portfolio['channels'].astype(str).str.contains('email').astype(int)
portfolio['mobile'] = portfolio['channels'].astype(str).str.contains('mobile').astype(int)
portfolio['social'] = portfolio['channels'].astype(str).str.contains('social').astype(int)

# 기존 channels 컬럼 제거
portfolio = portfolio.drop('channels', axis=1)

In [79]:
# portfolio 테이블의 필요없는 인덱스 컬럼 제거
portfolio = portfolio.drop('Unnamed: 0', axis=1)

---

# transcript

In [80]:
# 딕셔너리처럼 생긴 문자열을 진짜 딕셔너리로 변환
transcript['value'] = transcript['value'].apply(ast.literal_eval)

# 딕셔너리의 키 -> 새로운 컬럼
value_df = pd.DataFrame(transcript['value'].tolist())
transcript = pd.concat([transcript, value_df], axis=1)

# offer id를 offer_id로 컬럼명 통일
transcript['offer_id'] = transcript['offer_id'].fillna(transcript['offer id'])

# offer id 컬럼 제거
transcript = transcript.drop('offer id', axis=1)

# value 컬럼 제거
transcript = transcript.drop('value', axis=1)

---

# transcript x profile

In [81]:
# transcript 기준으로 profile 데이터를 left Join
merged_df = pd.merge(
    transcript,
    profile,
    left_on='person',
    right_on='id',
    how='left'
)

# 필요 없는 id 컬럼(person과 중복)은 버리기
merged_df = merged_df.drop(columns='id')

In [82]:
# 결측치 처리
# gender의 결측치 'Unknown'으로 채우기 
merged_df['gender'] = merged_df['gender'].fillna('Unknown')

# age의 118을 결측치(NaN)로 바꿔주기 
merged_df['age'] = merged_df['age'].replace(118, np.nan)

# income은 이미 결측치(NaN) 상태

---

# all_merged

In [83]:
all_merged_df = pd.merge(
    merged_df,
    portfolio,
    left_on='offer_id',
    right_on='id',
    how='left'
)

all_merged_df = all_merged_df.drop(columns='id')

# reward 컬럼명 변경(명확하게)
all_merged_df = all_merged_df.rename(columns={
    "reward_x": "transcript_reward",
    "reward_y": "portfolio_reward"
})

In [84]:
# offer_id 이름 변경 (쿠폰명_difficulty_reward_duration)
portfolio['offer_name'] = (
    portfolio['offer_type'] + '_' + 
    portfolio['difficulty'].astype(str) + '_' + 
    portfolio['reward'].astype(str) + '_' + 
    portfolio['duration'].astype(str)
)
# id : key, offer_name : value
offer_name_dict = portfolio.set_index('id')['offer_name'].to_dict()
all_merged_df['offer_id'] = all_merged_df['offer_id'].map(offer_name_dict)


# 사람(person)별로 먼저 묶고, 그 안에서 시간(time) 순서대로 오름차순 정렬
all_merged_df = all_merged_df.sort_values(by=['person', 'time', 'Unnamed: 0']) # - Unnamed: 0 순서 추가

In [85]:
check(all_merged_df, 'all_merged_df.csv')

all_merged_df.csv 기본 정보

[1] 데이터 크기


(306534, 19)


[2] 컬럼 정보
<class 'pandas.DataFrame'>
Index: 306534 entries, 55972 to 289924
Data columns (total 19 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Unnamed: 0         306534 non-null  int64         
 1   person             306534 non-null  str           
 2   event              306534 non-null  str           
 3   time               306534 non-null  int64         
 4   amount             138953 non-null  float64       
 5   offer_id           167581 non-null  str           
 6   transcript_reward  33579 non-null   float64       
 7   gender             306534 non-null  str           
 8   age                272762 non-null  float64       
 9   became_member_on   306534 non-null  datetime64[us]
 10  income             272762 non-null  float64       
 11  portfolio_reward   167581 non-null  float64       
 12  difficulty         167581 non-null  float64       
 13  duration           167581 non-null  float64  

,결측치 개수
transcript_reward,272955
amount,167581
offer_id,138953
mobile,138953
email,138953
social,138953
web,138953
offer_type,138953
duration,138953
difficulty,138953



[4] 중복 데이터 개수


np.int64(0)

---

# promotion

In [86]:
# 조건 1: 쿠폰 타입이 bogo, discount, informational 인 것
cond_offers = all_merged_df['offer_type'].isin(['bogo', 'discount', 'informational'])

# 조건 2: 이벤트 종류가 transaction(결제) 인 것
cond_transactions = all_merged_df['event'] == 'transaction'

target_df = all_merged_df[cond_offers | cond_transactions].copy()

print(target_df['offer_type'].value_counts(dropna=False))
print(target_df['event'].value_counts(dropna=False))
display(target_df.head())
display(target_df.shape)

offer_type
NaN              138953
bogo              71617
discount          69898
informational     26066
Name: count, dtype: int64
event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64


,Unnamed: 0,person,event,time,amount,offer_id,transcript_reward,gender,age,became_member_on,income,portfolio_reward,difficulty,duration,offer_type,web,email,mobile,social
55972,55972,0009655768c64bdeb2e877511632db8f,offer received,168,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
77705,77705,0009655768c64bdeb2e877511632db8f,offer viewed,192,NaN,informational_0_0_3,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,3.0,informational,0.0,1.0,1.0,1.0
89291,89291,0009655768c64bdeb2e877511632db8f,transaction,228,22.16,NaN,NaN,M,33.0,2017-04-21,72000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113605,113605,0009655768c64bdeb2e877511632db8f,offer received,336,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0
139992,139992,0009655768c64bdeb2e877511632db8f,offer viewed,372,NaN,informational_0_0_4,NaN,M,33.0,2017-04-21,72000.0,0.0,0.0,4.0,informational,1.0,1.0,1.0,0.0


(306534, 19)

In [87]:
# person당 offer_id를 하나의 행으로 설정하여, 흩어진 고객 행동의 순서를 보기 편하게 해주는 "피벗테이블 생성 코드"

# 1. 피벗을 돌릴 '쿠폰 이력서' 데이터만 빼내기
offers_df = target_df[target_df['event'] != 'transaction'].copy()

# 2. 안전한 금고에 보관할 '순수 영수증' 데이터만 빼내기
transactions_df = target_df[target_df['event'] == 'transaction'].copy()

print(f"피벗할 쿠폰 데이터: {len(offers_df)} 줄")
print(f"금고에 보관한 영수증: {len(transactions_df)} 줄")

피벗할 쿠폰 데이터: 167581 줄
금고에 보관한 영수증: 138953 줄


In [88]:
# 1. 시간 순서대로 예쁘게 줄 세우기
offers_df = offers_df.sort_values(['person', 'offer_id', 'time'])

# 2. 'received' 이벤트가 등장할 때마다 1, 아니면 0인 깃발(Flag) 만들기
offers_df['is_received'] = (offers_df['event'] == 'offer received').astype(int)

# 3. 사람과 쿠폰 단위로 묶어서, 깃발을 누적해서 더하기 (Cumsum)
offers_df['offer_cycle'] = offers_df.groupby(['person', 'offer_id'])['is_received'].cumsum()

# 4. 피벗 돌리기
pivot_df = offers_df.pivot_table(
    index=['person', 'offer_id', 'offer_cycle'],
    columns='event',
    values='time',
    aggfunc='min'
).reset_index()

pivot_df.columns.name = None
pivot_df = pivot_df[['person', 'offer_id', 'offer_cycle', 'offer received', 'offer viewed', 'offer completed']]

# reward 따로 뽑아서 merge <- 여기서 중복이 생기면서 초기 final_df와 차이가 생겼던 것
# reward_df = offers_df[offers_df['event'] == 'offer completed'][['person', 'offer_id', 'offer_cycle', 'transcript_reward']]
# pivot_df = pivot_df.merge(reward_df, on=['person', 'offer_id', 'offer_cycle'], how='left')

# portfolio 정보 붙이기
pivot_df = pivot_df.merge(
    portfolio[['offer_name', 'offer_type', 'difficulty', 'reward', 'duration', 'web', 'email', 'mobile', 'social']],
    left_on='offer_id',
    right_on='offer_name',
    how='left'
).drop(columns='offer_name')

# profile 정보 붙이기
pivot_df = pivot_df.merge(
    profile[['id', 'gender', 'age', 'became_member_on', 'income']],
    left_on='person',
    right_on='id',
    how='left'
).drop(columns='id')

display(pivot_df.head())
display(pivot_df.shape)

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,web,email,mobile,social,gender,age,became_member_on,income
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,1,408.0,456.0,414.0,bogo,5,5,5,1,1,1,1,M,33,2017-04-21,72000.0
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,1,504.0,540.0,528.0,discount,10,2,10,1,1,1,1,M,33,2017-04-21,72000.0
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,1,576.0,NaN,576.0,discount,10,2,7,1,1,1,0,M,33,2017-04-21,72000.0
3,0009655768c64bdeb2e877511632db8f,informational_0_0_3,1,168.0,192.0,NaN,informational,0,0,3,0,1,1,1,M,33,2017-04-21,72000.0
4,0009655768c64bdeb2e877511632db8f,informational_0_0_4,1,336.0,372.0,NaN,informational,0,0,4,1,1,1,0,M,33,2017-04-21,72000.0


(76277, 18)

In [89]:
# 1. 원본에서 offer_id와 offer_type 짝꿍 사전 만들기
offer_dict = offers_df[['offer_id', 'offer_type']].drop_duplicates().set_index('offer_id')['offer_type'].to_dict()

# 2. 피벗 테이블의 offer_id를 보고, 임시로 쿠폰 타입(bogo, discount)을 가져오기
temp_offer_type = pivot_df['offer_id'].map(offer_dict)

# 3. 기존 숫자였던 'offer_cycle' 컬럼 위에 곧바로 덮어쓰기
pivot_df['offer_cycle'] = temp_offer_type.str.capitalize() + '_' + pivot_df['offer_cycle'].astype(str)

In [90]:
# 피벗테이블에 amount 붙이기

# 1. 금고(transactions_df)에서 영수증 알맹이만 꺼내기
transactions_df = transactions_df[["person", "time", "amount"]]

# 2. 피벗 테이블(pivot_df)에 영수증(receipts) 1:1 도킹하기
final_df = pivot_df.merge(
    transactions_df,
    left_on=["person", "offer completed"],
    right_on=["person", "time"],
    how="left"
).drop(columns="time")

# 3. 태블로 퍼널 시각화를 위한 0/1 파생변수 생성
final_df["is_received"] = final_df["offer received"].notna().astype(int)
final_df["is_viewed"] = final_df["offer viewed"].notna().astype(int)
final_df["is_completed"] = final_df["offer completed"].notna().astype(int)

# 4. 정상 흐름 플래그
final_df["is_normal_flow"] = (
    final_df["offer received"].notna() &
    final_df["offer viewed"].notna() &
    final_df["offer completed"].notna() &
    (final_df["offer received"] <= final_df["offer viewed"]) &
    (final_df["offer viewed"] <= final_df["offer completed"])
).astype(int)

# 5. 더블카운팅 처리 플래그
final_df["tx_key"] = final_df["person"].astype(str) + "_" + final_df["offer completed"].astype(str)
final_df = final_df.sort_values(by=["tx_key", "offer viewed"], ascending=[True, False])
final_df["is_deduplicated"] = (
    ~final_df.duplicated(subset=["tx_key"], keep="first") |
    final_df["offer completed"].isna()
).astype(int)
final_df = final_df.drop(columns="tx_key")

# 6. flow_type 추가 (전체 흐름 분류)
def get_flow_type(row):
    is_viewed = row["is_viewed"]
    is_completed = row["is_completed"]
    viewed = row["offer viewed"]
    completed = row["offer completed"]

    if is_viewed == 1 and is_completed == 1:
        if viewed <= completed:
            return 1  # 받기 -> 보기 -> 완료 (정상)
        else:
            return 0  # 받기 -> 완료 -> 보기 (비정상)
    elif is_viewed == 0 and is_completed == 1:
        return 2      # 받기 -> 완료
    elif is_viewed == 1 and is_completed == 0:
        return 3      # 받기 -> 보기
    else:
        return 4      # 받기만

final_df["flow_type"] = final_df.apply(get_flow_type, axis=1)

# 7. 정상흐름(flow_type == 1)에 대해 누적 amount 계산
tx = transactions_df[["person", "time", "amount"]].copy()
tx_dict = {
    person: group.sort_values("time")
    for person, group in tx.groupby("person")
}

normal_mask = final_df["flow_type"] == 1
normal_df = final_df[normal_mask].copy()

# cycle 번호 추출
normal_df["cycle_num"] = normal_df["offer_cycle"].str.extract(r"_(\d+)$").astype(int)

# final_df 전체 기준으로 가장 첫 번째 received 시점 (cycle 1 재수령 케이스 대응)
first_received = (
    final_df.groupby(["person", "offer_id"])["offer received"]
    .min()
    .reset_index()
    .rename(columns={"offer received": "first_received"})
)
normal_df = normal_df.merge(first_received, on=["person", "offer_id"], how="left")

results = []
for _, row in normal_df.iterrows():
    person_tx = tx_dict.get(row["person"], pd.DataFrame(columns=["time", "amount"]))

    # cycle 1 -> first_received 기준, cycle 2 이상 -> 해당 cycle의 offer received 기준
    received_start = row["first_received"] if row["cycle_num"] == 1 else row["offer received"]

    amt_received = person_tx[
        (person_tx["time"] >= received_start) &
        (person_tx["time"] <= row["offer completed"])
    ]["amount"].sum()

    amt_viewed = person_tx[
        (person_tx["time"] >= row["offer viewed"]) &
        (person_tx["time"] <= row["offer completed"])
    ]["amount"].sum()

    results.append({
        "amount_from_received": amt_received,
        "amount_from_viewed": amt_viewed
    })

result_df = pd.DataFrame(results, index=normal_df.index)
normal_df = pd.concat([normal_df, result_df], axis=1)
normal_df = normal_df.drop(columns=["first_received", "cycle_num"])

# 8. final_df에 amount_from_received, amount_from_viewed 추가 (정상흐름만, 나머지 NaN)
final_df["amount_from_received"] = None
final_df["amount_from_viewed"] = None
final_df.loc[normal_df.index, "amount_from_received"] = normal_df["amount_from_received"].values
final_df.loc[normal_df.index, "amount_from_viewed"] = normal_df["amount_from_viewed"].values

display(final_df.head())
display(final_df.shape)
print(f"flow_type 분포:")
print(final_df["flow_type"].value_counts().sort_index())
print(f"amount_from_received < difficulty (정상흐름): {(final_df[final_df["flow_type"]==1]["amount_from_received"] < final_df[final_df["flow_type"]==1]["difficulty"]).sum():,}")
print(f"amount_from_viewed < difficulty (정상흐름): {(final_df[final_df["flow_type"]==1]["amount_from_viewed"] < final_df[final_df["flow_type"]==1]["difficulty"]).sum():,}")


,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated,flow_type,amount_from_received,amount_from_viewed
0,0009655768c64bdeb2e877511632db8f,bogo_5_5_5,Bogo_1,408.0,456.0,414.0,bogo,5,5,5,...,72000.0,8.57,1,1,1,0,1,0,11.93,11.93
1,0009655768c64bdeb2e877511632db8f,discount_10_2_10,Discount_1,504.0,540.0,528.0,discount,10,2,10,...,72000.0,14.11,1,1,1,0,1,0,22.05,22.05
2,0009655768c64bdeb2e877511632db8f,discount_10_2_7,Discount_1,576.0,NaN,576.0,discount,10,2,7,...,72000.0,10.27,1,0,1,0,1,2,22.05,22.05
9,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,discount,7,3,7,...,57000.0,11.93,1,1,1,1,1,1,13.44,13.44
7,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,bogo,5,5,7,...,57000.0,22.05,1,1,1,1,1,1,10.32,10.32


(76277, 27)

flow_type 분포:
flow_type
0     4205
1    23267
2     5629
3    30253
4    12923
Name: count, dtype: int64
amount_from_received < difficulty (정상흐름): 626
amount_from_viewed < difficulty (정상흐름): 739


In [91]:
print(f"is_deduplicated == 1 (유효한 행): {(final_df['is_deduplicated'] == 1).sum():,}")
print(f"is_deduplicated == 0 (제거된 행): {(final_df['is_deduplicated'] == 0).sum():,}")
print(f"전체: {len(final_df):,}")

is_deduplicated == 1 (유효한 행): 73,719
is_deduplicated == 0 (제거된 행): 2,558
전체: 76,277


In [92]:
print(f"전체 is_deduplicated == 1: {(final_df['is_deduplicated'] == 1).sum():,}")
print(f"그 중 offer completed 있는 행: {((final_df['is_deduplicated'] == 1) & final_df['offer completed'].notna()).sum():,}")
print(f"제거된 더블카운팅 수: {(final_df['is_deduplicated'] == 0).sum():,}")

전체 is_deduplicated == 1: 73,719
그 중 offer completed 있는 행: 30,543
제거된 더블카운팅 수: 2,558


In [93]:
both = (final_df['is_normal_flow'] == 1) & (final_df['is_deduplicated'] == 1)
print(f"정상 흐름 + 더블카운팅 제거: {both.sum():,}")

정상 흐름 + 더블카운팅 제거: 21,954


In [94]:
check(final_df, 'promotion_df.csv')

promotion_df.csv 기본 정보

[1] 데이터 크기


(76277, 27)


[2] 컬럼 정보
<class 'pandas.DataFrame'>
Index: 76277 entries, 0 to 76266
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   person                76277 non-null  str           
 1   offer_id              76277 non-null  str           
 2   offer_cycle           76277 non-null  str           
 3   offer received        76277 non-null  float64       
 4   offer viewed          57725 non-null  float64       
 5   offer completed       33101 non-null  float64       
 6   offer_type            76277 non-null  str           
 7   difficulty            76277 non-null  int64         
 8   reward                76277 non-null  int64         
 9   duration              76277 non-null  int64         
 10  web                   76277 non-null  int64         
 11  email                 76277 non-null  int64         
 12  mobile                76277 non-null  int64         
 13  social               

,결측치 개수
amount_from_viewed,53010
amount_from_received,53010
offer completed,43176
amount,43176
offer viewed,18552
gender,9776
income,9776
offer_cycle,0
offer_id,0
offer received,0



[4] 중복 데이터 개수


np.int64(0)

---

# 확인

더블 카운팅이 정상 흐름에서만 처리가 된 것 같아서 다른 흐름에서의 더블 카운팅도 체크해봐야할듯

In [95]:
promotion = pd.read_csv('../data/promotion_df.csv')

In [96]:
print(f"전체 행 수: {len(promotion):,}")
print(f"정상 흐름(is_normal_flow) 행 수: {promotion['is_normal_flow'].sum():,}")
print(f"더블카운팅 제거 후(is_deduplicated) 행 수: {promotion['is_deduplicated'].sum():,}")

both = (promotion['is_normal_flow'] == 1) & (promotion['is_deduplicated'] == 1)
print(f"정상 흐름 + 더블카운팅 제거: {both.sum():,}")

print(f"차이: {(promotion['is_normal_flow'].sum() - promotion['is_deduplicated'].sum()):,}")

전체 행 수: 76,755
정상 흐름(is_normal_flow) 행 수: 23,519
더블카운팅 제거 후(is_deduplicated) 행 수: 22,333
정상 흐름 + 더블카운팅 제거: 22,333
차이: 1,186


In [97]:
print(f"전체 행 수: {len(final_df):,}")
print(f"정상 흐름(is_normal_flow) 행 수: {final_df['is_normal_flow'].sum():,}")
print(f"더블카운팅 제거 후(is_deduplicated) 행 수: {final_df['is_deduplicated'].sum():,}")

both = (final_df['is_normal_flow'] == 1) & (final_df['is_deduplicated'] == 1)
print(f"정상 흐름 + 더블카운팅 제거: {both.sum():,}")

print(f"차이: {(final_df['is_normal_flow'].sum() - final_df['is_deduplicated'].sum()):,}")

전체 행 수: 76,277
정상 흐름(is_normal_flow) 행 수: 23,267
더블카운팅 제거 후(is_deduplicated) 행 수: 73,719
정상 흐름 + 더블카운팅 제거: 21,954
차이: -50,452


정상 흐름만 먼저 뽑아내고 거기서 더블 카운팅을 제거해본다면?

In [98]:
# 1) 두 조건의 개수 확인
count_true_conversion = promotion['is_normal_flow'].sum()
count_valid_attr = promotion['is_deduplicated'].sum()
diff = count_true_conversion - count_valid_attr

print(f"is_true_conversion 합계: {count_true_conversion:,}")
print(f"valid_attribution_flag 합계: {count_valid_attr:,}")
print(f"차이: {diff:,}")

# 2) is_true_conversion=1 이지만 valid_attribution_flag=0 인 행만 추출
# - last touch로 선택되지 못한 값들 선택
lost_rows = promotion[
    (promotion['is_normal_flow'] == 1) &
    (promotion['is_deduplicated'] == 0)
].copy()

print(f"전환으로는 잡혔지만 최종 귀속에서는 빠진 행 수: {len(lost_rows):,}")

# 3) 검증
assert len(lost_rows) == diff, "차이 수와 실제 빠진 행 수가 다름"

display(lost_rows.head())

is_true_conversion 합계: 23,519
valid_attribution_flag 합계: 22,333
차이: 1,186
전환으로는 잡혔지만 최종 귀속에서는 빠진 행 수: 1,186


,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,transcript_reward,offer_type,difficulty,reward,...,gender,age,became_member_on,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated
5,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,5.0,discount,20,5,...,O,40,2018-01-09,57000.0,22.05,1,1,1,1,0
48,00ae03011f9f49b8a4b3e6d416678b0b,bogo_10_10_7,Bogo_2,504.0,534.0,618.0,10.0,bogo,10,10,...,M,55,2015-11-15,83000.0,30.83,1,1,1,1,0
81,00cf1bbec83f4a658f8994e556db4633,discount_10_2_10,Discount_2,336.0,438.0,564.0,2.0,discount,10,2,...,M,53,2018-05-15,73000.0,33.28,1,1,1,1,0
153,015c3d28c67e46aa95e9ec97c27220e8,discount_10_2_7,Discount_1,504.0,510.0,618.0,2.0,discount,10,2,...,M,56,2016-03-22,99000.0,22.12,1,1,1,1,0
160,01633b71b3a2457aa7d35d8bcc3afb5a,discount_20_5_10,Discount_1,504.0,552.0,600.0,5.0,discount,20,5,...,M,62,2015-09-02,90000.0,24.76,1,1,1,1,0


---

# normal_flow

In [99]:
# # 정상 흐름 필터링: received <= viewed <= completed (모두 존재해야 함)
# normal_flow_df = final_df[
#     final_df['offer received'].notna() &
#     final_df['offer viewed'].notna() &
#     final_df['offer completed'].notna() &
#     (final_df['offer received'] <= final_df['offer viewed']) &
#     (final_df['offer viewed'] <= final_df['offer completed'])
# ].copy()

# display(normal_flow_df.head())
# display(normal_flow_df.shape)

In [100]:
# normal_flow_df.to_csv('../data/normal_flow_df.csv', index=False)

In [101]:
# check(normal, 'normal_flow_df.csv')

---

In [102]:
# def get_flow_type(row):
#     viewed = row['offer viewed']
#     completed = row['offer completed']
#     is_viewed = row['is_viewed']
#     is_completed = row['is_completed']

#     if is_viewed == 1 and is_completed == 1:
#         if pd.isna(viewed) or pd.isna(completed):
#             return 0  # 타임스탬프 없으면 비정상
#         elif viewed <= completed:
#             return 1  # 받기 -> 보기 -> 완료 (정상)
#         else:
#             return 0  # 받기 -> 완료 -> 보기 (비정상)
#     elif is_viewed == 0 and is_completed == 1:
#         return 2      # 받기 -> 완료
#     elif is_viewed == 1 and is_completed == 0:
#         return 3      # 받기 -> 보기
#     else:
#         return 4      # 받기만

# promotion['flow_type'] = promotion.apply(get_flow_type, axis=1)

# # 분포 확인
# print(promotion['flow_type'].value_counts().sort_index())
def get_flow_type(row):
    viewed = row['offer viewed']
    completed = row['offer completed']
    is_viewed = row['is_viewed']
    is_completed = row['is_completed']

    if is_viewed == 1 and is_completed == 1:
        if viewed <= completed:
            return 1  # 받기 -> 보기 -> 완료 (정상)
        else:
            return 0  # 받기 -> 완료 -> 보기 (비정상)
    elif is_viewed == 0 and is_completed == 1:
        return 2      # 받기 -> 완료
    elif is_viewed == 1 and is_completed == 0:
        return 3      # 받기 -> 보기
    else:
        return 4      # 받기만

promotion['flow_type'] = promotion.apply(get_flow_type, axis=1)
counts = promotion['flow_type'].value_counts().sort_index()

print(counts)
print(f'\n총 합계: {counts.sum()}')

flow_type
0     4318
1    23519
2     5742
3    30253
4    12923
Name: count, dtype: int64

총 합계: 76755


---

# 누적 amount 계산

정상 흐름(is_normal_flow == 1)에 대해 person별로 묶고 time 순 정렬 후,
각 쿠폰(offer_id)별로 두 가지 누적 amount를 계산합니다.
- **amount_from_received**: offer received ~ offer completed 사이 트랜잭션 합산
- **amount_from_viewed**: offer viewed ~ offer completed 사이 트랜잭션 합산

In [103]:
# # person별로 transactions_df를 미리 딕셔너리로 그룹화 (속도 최적화)
# tx = transactions_df[["person", "time", "amount"]].copy()
# tx_dict = {
#     person: group.sort_values("time")
#     for person, group in tx.groupby("person")
# }

# # 정상 흐름만 필터링
# normal_df = final_df[final_df["is_normal_flow"] == 1].copy()

# results = []

# for _, row in normal_df.iterrows():
#     person_tx = tx_dict.get(row["person"], pd.DataFrame(columns=["time", "amount"]))

#     # received ~ completed 사이 누적합
#     amt_received = person_tx[
#         (person_tx["time"] >= row["offer received"]) &
#         (person_tx["time"] <= row["offer completed"])
#     ]["amount"].sum()

#     # viewed ~ completed 사이 누적합
#     amt_viewed = person_tx[
#         (person_tx["time"] >= row["offer viewed"]) &
#         (person_tx["time"] <= row["offer completed"])
#     ]["amount"].sum()

#     results.append({
#         "amount_from_received": amt_received,
#         "amount_from_viewed": amt_viewed
#     })

# result_df = pd.DataFrame(results, index=normal_df.index)
# normal_df['amount_from_received'] = result_df['amount_from_received']
# normal_df['amount_from_viewed'] = result_df['amount_from_viewed']
# normal_df = normal_df.drop(columns=['first_received', 'cycle_num'], errors='ignore')

# display(normal_df[["person", "offer_id", "offer received", "offer viewed", "offer completed",
#                     "difficulty", "amount_from_received", "amount_from_viewed"]].head(10))
# print(f"amount_from_received < difficulty인 케이스: {(normal_df["amount_from_received"] < normal_df["difficulty"]).sum():,}")
# print(f"amount_from_viewed < difficulty인 케이스: {(normal_df["amount_from_viewed"] < normal_df["difficulty"]).sum():,}")

받기 -> 완료 사이

amount_from_received < difficulty (76건)
- 쿠폰을 받은 시점부터 완료까지 누적 금액이 difficulty에 못 미치는데 completed 처리된 케이스
- 이건 진짜 이상한 케이스 — 조건 자체를 못 채웠는데 완료됨

amount_from_viewed < difficulty (684건)
- viewed 이후 누적 금액은 difficulty에 못 미치지만, received 이후로는 채운 케이스
- 즉 쿠폰을 보기 전에 이미 쓴 금액이 difficulty 달성에 기여한 것
- 이게 바로 처음에 얘기했던 누적 완성 이슈 — viewed 이전 거래가 complete 조건에 포함된 거예요

결론적으로:
76건은 데이터 자체의 이상치 (어떤 기준으로도 조건 미달인데 완료)<br>
684건은 viewed 이전 금액이 difficulty를 채우는 데 쓰인 케이스 → 우리가 찾던 누적 흐름 이슈

In [104]:
# # 76건 추출
# weird_df = normal_df[normal_df['amount_from_received'] < normal_df['difficulty']].copy()

# print(f"총 {len(weird_df)}건")
# print()

# # offer_id별 분포
# print("=== offer_id별 건수 ===")
# display(weird_df['offer_id'].value_counts())

# print()

# # 주요 컬럼만 확인
# display(weird_df[[
#     'person', 'offer_id', 
#     'offer received', 'offer viewed', 'offer completed',
#     'difficulty', 'amount_from_received', 'amount_from_viewed'
# ]].sort_values('amount_from_received'))

In [105]:
# # promotion_df 전체에서 해당 person의 해당 offer_id 모든 cycle 확인
# person_id = '2ac20da42e704b4bae0a799018096259'
# offer = 'discount_7_3_7'

# display(normal_df[
#     (normal_df['person'] == person_id) & 
#     (normal_df['offer_id'] == offer)
# ][['person', 'offer_id', 'offer_cycle', 'offer received', 'offer viewed', 'offer completed', 'difficulty', 'amount_from_received']])

데이터에서 봤을 땐 같은 쿠폰을 비슷한 시기에 받았을 때 문제가 발생한다

1. 트랜잭션이 쿠폰 유효기간 밖에서 발생
    - duration 기간을 넘긴 트랜잭션은 집계에서 빠졌을 수 있는데, 실제 스타벅스 로직은 그걸 포함해서 계산했을 수도 있어요

2. 아까 말한 중복 집계 문제의 반대 케이스
    - 다른 쿠폰의 트랜잭션이 이 쿠폰에 귀속됐을 수 있어요
    - 즉 스타벅스 내부 로직은 여러 쿠폰에 걸친 거래를 어떻게든 하나로 귀속시켰는데, 우리는 그 로직을 모르는 것

3. 데이터 자체의 누락
    - transcript 원본에 일부 트랜잭션이 아예 빠져있을 가능성
    - 샘플 데이터라면 특히 그럴 수 있어요

4. offer completed 이벤트 타이밍 문제
    - offer completed가 트랜잭션과 같은 time에 찍히는데, 혹시 completed 판정이 해당 트랜잭션 이전에 이미 내부적으로 결정됐을 수도 있어요

---

In [106]:
# 저장
# normal_df.to_csv("../data/normal_flow_cumulative.csv", index=False)

In [107]:
# all_merged_df.to_csv("../data/all_merged.csv", index=False)
final_df.to_csv("../data/promotion_df_final.csv", index=False)
print("저장 완료! -> ../data/promotion_df_final.csv")
print(f"shape: {final_df.shape}")
print(f"columns: {final_df.columns.tolist()}")


저장 완료! -> ../data/promotion_df_final.csv
shape: (76277, 27)
columns: ['person', 'offer_id', 'offer_cycle', 'offer received', 'offer viewed', 'offer completed', 'offer_type', 'difficulty', 'reward', 'duration', 'web', 'email', 'mobile', 'social', 'gender', 'age', 'became_member_on', 'income', 'amount', 'is_received', 'is_viewed', 'is_completed', 'is_normal_flow', 'is_deduplicated', 'flow_type', 'amount_from_received', 'amount_from_viewed']


In [108]:
normal_df.head()

,person,offer_id,offer_cycle,offer received,offer viewed,offer completed,offer_type,difficulty,reward,duration,...,income,amount,is_received,is_viewed,is_completed,is_normal_flow,is_deduplicated,flow_type,amount_from_received,amount_from_viewed
0,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,168.0,186.0,252.0,discount,7,3,7,...,57000.0,11.93,1,1,1,1,1,1,11.93,11.93
1,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,504.0,516.0,576.0,bogo,5,5,7,...,57000.0,22.05,1,1,1,1,1,1,22.05,22.05
2,0011e0d4e6b944f998e987f904e8c1e5,discount_20_5_10,Discount_1,408.0,432.0,576.0,discount,20,5,10,...,57000.0,22.05,1,1,1,1,0,1,22.05,22.05
3,0020c2b971eb4e9188eac86d93036a77,bogo_10_10_5,Bogo_1,408.0,426.0,510.0,bogo,10,10,5,...,90000.0,17.24,1,1,1,1,1,1,17.24,17.24
4,0020c2b971eb4e9188eac86d93036a77,discount_10_2_10,Discount_1,0.0,12.0,54.0,discount,10,2,10,...,90000.0,17.63,1,1,1,1,1,1,17.63,17.63


In [109]:
print('완2')

완2
